# Irish ABC tune fine-tuning (Qwen3.5)

Fine-tune on your ABC tune CSV, then generate one in a chosen key/meter/length.
Run cells top to bottom. Needs a GPU runtime.

# 1. Prerequisites

We have to prepare our environement by installing and importing all the required libraries as well as initializing the constnats.

In [15]:
!pip install -q transformers peft bitsandbytes accelerate datasets ipywidgets music21 peft torch


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [3]:
from datasets import load_dataset
import torch
from peft import LoraConfig, get_peft_model, PeftModel
from music21 import converter
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
    DataCollatorForLanguageModeling, Trainer, TrainingArguments,
)

import ipywidgets as widgets
from IPython.display import display

DATASET_FILE="irishman_clean.csv"
FRACTION_OF_DATASET_TO_USE = 0.005
BASE_MODEL = "Qwen/Qwen3.5-0.8B"
OUTPUT = "lora-out"

# 2. Training the model

This step consists of preparing the data, loading the model and finetuning it.

In our case the model has not been finetuned on a large subset of the dataset due to the performance constraints.

## 2.1. Load your dataset

Expects a CSV with columns: `abc_notation, control_code, meter, key, note_length`.
Upload it to the Colab file browser (or mount Drive) and set the path below.

In [32]:
dataset = load_dataset("csv", data_files=DATASET_FILE, split="train")
dataset = dataset.shuffle(seed=42).select(range(int(FRACTION_OF_DATASET_TO_USE * len(dataset)))) # To take a part of the dataset cuz my ass will burn if I train it on the full thing. Cordialement

dataset = dataset.rename_column("abc_notation", "text")
dataset

Dataset({
    features: ['text', 'control_code', 'meter', 'key', 'note_length'],
    num_rows: 1070
})

## 2.2. Load base model + LoRA

In [33]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16),
    device_map="auto",
    trust_remote_code=True,
)

model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=32, task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
))
model.print_trainable_parameters()
print(f'Running on {model.device}')

Loading weights: 100%|██████████| 320/320 [00:01<00:00, 220.60it/s]


trainable params: 6,389,760 || all params: 758,782,784 || trainable%: 0.8421
Running on mps:0


## 2.3. Fine-tuning

In [34]:
tokenized = dataset.map(
    lambda x: tokenizer(x["text"], truncation=True, max_length=512, padding="max_length"),
    batched=True,
)

trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir="lora-out",
        per_device_train_batch_size=2,
        num_train_epochs=1,
        learning_rate=2e-4,
        logging_steps=50,
        save_strategy="epoch",
        bf16=True,
        report_to="none",
    ),
    train_dataset=tokenized,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)
trainer.train()

model.save_pretrained(OUTPUT)
tokenizer.save_pretrained(OUTPUT)

Map: 100%|██████████| 1070/1070 [00:00<00:00, 7744.64 examples/s]
/Users/andreeaioanaflorea/PycharmProjects/Project-ByteToNote/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss



KeyboardInterrupt



# 3. Run the model !

## 3.1. Load the model that we have trained before from disk

Obviously, we will do that only if we are starting cold and haven't already trained the model in this run 😉

In [4]:
tokenizer = AutoTokenizer.from_pretrained(OUTPUT)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16),
    device_map="auto",
    trust_remote_code=True,
)

model = PeftModel.from_pretrained(base_model, OUTPUT)
model.eval()

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3_5ForCausalLM(
      (model): Qwen3_5TextModel(
        (embed_tokens): Embedding(248320, 1024)
        (layers): ModuleList(
          (0-2): 3 x Qwen3_5DecoderLayer(
            (linear_attn): Qwen3_5GatedDeltaNet(
              (conv1d): Conv1d(6144, 6144, kernel_size=(4,), stride=(1,), padding=(3,), groups=6144, bias=False)
              (norm): Qwen3_5RMSNormGated()
              (out_proj): Linear4bit(in_features=2048, out_features=1024, bias=False)
              (in_proj_qkv): Linear4bit(in_features=1024, out_features=6144, bias=False)
              (in_proj_z): Linear4bit(in_features=1024, out_features=2048, bias=False)
              (in_proj_b): Linear4bit(in_features=1024, out_features=16, bias=False)
              (in_proj_a): Linear4bit(in_features=1024, out_features=16, bias=False)
            )
            (mlp): Qwen3_5MLP(
              (gate_proj): lora.Linear4bit(
                (base_layer): Linear4b

## 3.2. Let's get to the generation

### 3.2.1. We define a helper function

It will generate the tune, make sure it is clean from any artifacts our LLM might have left and play it 🎶🎶

In [5]:
# Those values usually work 😅
# key = "C"
# meter = "2/4"
# note_length = "1/4"

def generate(key, meter, note_length):
    max_bars = 16
    prompt = f"X:1\nL:{note_length}\nM:{meter}\nK:{key}\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    out = model.generate(
        **inputs,
        max_new_tokens=500,
        do_sample=True,
        temperature=0.65,
        top_p=0.90,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id
    )
    text = tokenizer.decode(out[0], skip_special_tokens=True)

    header_len = len(prompt)
    header = text[:header_len]
    body = text[header_len:]

    for end_marker in ['|]', '||']:
        if end_marker in body:
            body = body.split(end_marker)[0] + end_marker
            break

    parts = body.split("|")
    if len(parts) > max_bars + 1:
        kept = parts[: max_bars + 1]
        body = "|".join(kept)
        body = body.strip()

        if not body.endswith((']', '|', '||')):
            body += "|]"
    elif not body.strip().endswith((']', '|', '||')):
        body = body.strip() + "|]"

    final = header + body

    print(final)

    parsed_score = converter.parse(final, format='abc')
    parsed_score.show('midi')

### 3.2.2 Generation widget

A small eye-candy to help us change parameters for each generation and have the minimal interface in our notebook

In [6]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Create input widgets
key_input = widgets.Text(description="Key :")
meter_input = widgets.Text(description="Meter :")
note_length_input = widgets.Text(description="Note length :")

run_button = widgets.Button(description="Run", button_style="success")
output = widgets.Output()

def on_button_click(b):
    with output:
        clear_output()
        key = key_input.value
        meter = meter_input.value
        note_length = note_length_input.value

        print(f"Processing... Key: {key} | Meter: {meter} | Note length: {note_length}")
        generate(key, meter, note_length)

run_button.on_click(on_button_click)

display(key_input, meter_input, note_length_input, run_button, output)

Text(value='', description='Key :')

Text(value='', description='Meter :')

Text(value='', description='Note length :')

Button(button_style='success', description='Run', style=ButtonStyle())

Output()